# Fashion-MNIST Image Classification with a CNN

This notebook trains a small Convolutional Neural Network (CNN) to recognize clothing images. You can run it in Google Colab and view the graphs and predictions. No local setup is required.

**How to use:** choose **Runtime → Run all**. The first run downloads Fashion-MNIST.

## 1. Import libraries and set up the project

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 0
IMAGE_SIZE = 16
BATCH_SIZE = 100
EPOCHS = 5
LEARNING_RATE = 0.1

torch.manual_seed(SEED)
print('PyTorch version:', torch.__version__)
print('Device: CPU (this small model can run without a GPU)')

## 2. Download and prepare the Fashion-MNIST dataset

Each image is converted to a tensor and resized to 16 × 16 pixels. The dataset has 10 clothing categories.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f'Training images: {len(train_data):,}')
print(f'Test images: {len(test_data):,}')

### Preview a few images

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for i, ax in enumerate(axes):
    image, label = train_data[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[label])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Build the CNN model

The model uses two convolution layers to learn image features, max-pooling to reduce image size, and a final layer that scores the 10 clothing categories.

In [ ]:
class CNN(nn.Module):
    def __init__(self, out_1=16, out_2=32, number_of_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, out_1, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(out_1, out_2, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.fc = nn.Linear(out_2 * 4 * 4, number_of_classes)

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
print(model)

## 4. Train the model and check accuracy

An epoch is one pass through the training data. The loss shows how far the model's scores are from the correct labels; accuracy shows the share of test images classified correctly.

In [ ]:
cost_list = []
accuracy_list = []
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    cost_list.append(total_loss)

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
    accuracy = correct / len(test_data)
    accuracy_list.append(accuracy)
    print(f'Epoch {epoch + 1}/{EPOCHS} | Loss sum: {total_loss:.2f} | Test accuracy: {accuracy:.2%}')

print(f'Training time: {time.time() - start_time:.1f} seconds')

## 5. Visualize the training results

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, cost_list, marker='o', label='Loss sum')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Sum of batch losses')
ax2 = ax1.twinx()
ax2.plot(epochs, accuracy_list, marker='s', label='Test accuracy')
ax2.set_ylabel('Test accuracy')
plt.title('Fashion-MNIST CNN: training progress')
fig.tight_layout()
plt.show()

## 6. View model predictions

The title above each image shows the model's prediction and the correct label.

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    predictions = model(images).argmax(dim=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f'Pred: {CLASS_NAMES[predictions[i]]}\nTrue: {CLASS_NAMES[labels[i]]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Summary

This project uses PyTorch to train a CNN for a 10-class image classification task. The notebook shows example images, training loss, test accuracy, and sample predictions. Your results may vary slightly depending on the runtime and training settings.

**Dataset:** [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)